# DBSCAN from Scratch

**Goal:** Implement DBSCAN (Density-Based Spatial Clustering of Applications with Noise) from first principles, with intuition for when it beats K-means.

---

## Core Intuition

DBSCAN groups together points that are **densely packed** and labels sparse points as noise. Unlike K-means, it:
- Does NOT require specifying K (number of clusters)
- Can find **arbitrarily shaped** clusters
- Has a built-in notion of **noise/outliers**

**Key concepts:**
- **Epsilon-neighborhood:** $N_\varepsilon(x) = \{y : \|x - y\| \leq \varepsilon\}$
- **Core point:** Has at least `min_samples` neighbors within $\varepsilon$
- **Border point:** Within $\varepsilon$ of a core point but not core itself
- **Noise point:** Neither core nor border

**Algorithm:**
1. For each unvisited point, check if it's a core point
2. If core: start a new cluster and expand by adding all density-reachable points
3. If not core: mark as noise (may later be claimed as border)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. DBSCAN Implementation

In [ ]:
class DBSCAN:
    """DBSCAN clustering from scratch."""
    
    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels_ = None
        self.core_sample_indices_ = None
        self.n_clusters_ = 0
    
    def _compute_distances(self, X):
        """Compute pairwise Euclidean distance matrix."""
        sq = np.sum(X ** 2, axis=1)
        dist_sq = sq[:, np.newaxis] + sq[np.newaxis, :] - 2 * X @ X.T
        return np.sqrt(np.maximum(dist_sq, 0))
    
    def _get_neighbors(self, distances, point_idx):
        """Get indices of points within epsilon of point_idx."""
        return np.where(distances[point_idx] <= self.eps)[0]
    
    def _expand_cluster(self, distances, point_idx, neighbors, cluster_id, labels, visited):
        """Expand cluster from a core point by density-reachability."""
        labels[point_idx] = cluster_id
        
        # Use a queue (list) for BFS expansion
        queue = list(neighbors)
        i = 0
        while i < len(queue):
            neighbor_idx = queue[i]
            i += 1
            
            if not visited[neighbor_idx]:
                visited[neighbor_idx] = True
                new_neighbors = self._get_neighbors(distances, neighbor_idx)
                
                # If this neighbor is also a core point, add its neighbors to the queue
                if len(new_neighbors) >= self.min_samples:
                    for nn in new_neighbors:
                        if nn not in queue[:i]:  # avoid revisiting
                            queue.append(nn)
            
            # Assign to cluster if not yet assigned
            if labels[neighbor_idx] == -1:
                labels[neighbor_idx] = cluster_id
    
    def fit(self, X):
        n = X.shape[0]
        distances = self._compute_distances(X)
        
        self.labels_ = np.full(n, -1)  # -1 = noise
        visited = np.zeros(n, dtype=bool)
        cluster_id = 0
        
        # Identify core points
        neighbor_counts = np.sum(distances <= self.eps, axis=1)  # includes self
        self.core_sample_indices_ = np.where(neighbor_counts >= self.min_samples)[0]
        is_core = np.zeros(n, dtype=bool)
        is_core[self.core_sample_indices_] = True
        
        for point_idx in range(n):
            if visited[point_idx]:
                continue
            
            visited[point_idx] = True
            neighbors = self._get_neighbors(distances, point_idx)
            
            if len(neighbors) < self.min_samples:
                # Not a core point, mark as noise (may be reclaimed as border)
                self.labels_[point_idx] = -1
            else:
                # Core point: start new cluster
                self._expand_cluster(distances, point_idx, neighbors, cluster_id, self.labels_, visited)
                cluster_id += 1
        
        self.n_clusters_ = cluster_id
        self.is_core_ = is_core
        return self

# Quick test
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

db = DBSCAN(eps=0.2, min_samples=5)
db.fit(X_moons)

print(f"Number of clusters: {db.n_clusters_}")
print(f"Number of noise points: {np.sum(db.labels_ == -1)}")
print(f"Number of core points: {len(db.core_sample_indices_)}")

## 2. Visualize Core, Border, and Noise Points

In [ ]:
def plot_dbscan(X, db, title, ax=None):
    """Visualize DBSCAN results with core/border/noise distinction."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    labels = db.labels_
    unique_labels = set(labels)
    n_clusters = len(unique_labels - {-1})
    
    cmap = plt.cm.Set1
    colors_arr = [cmap(i / max(n_clusters, 1)) for i in range(n_clusters)]
    
    for k in unique_labels:
        if k == -1:
            # Noise
            mask = labels == k
            ax.scatter(X[mask, 0], X[mask, 1], c='gray', marker='x', s=30, alpha=0.5, label='Noise')
        else:
            mask = labels == k
            core_mask = mask & db.is_core_
            border_mask = mask & ~db.is_core_
            
            color = colors_arr[k % len(colors_arr)]
            ax.scatter(X[core_mask, 0], X[core_mask, 1], c=[color], s=50, alpha=0.8,
                       edgecolors='black', linewidths=0.5, label=f'Core (C{k})')
            ax.scatter(X[border_mask, 0], X[border_mask, 1], c=[color], s=20, alpha=0.4,
                       marker='s', label=f'Border (C{k})')
    
    ax.set_title(f'{title}\n{n_clusters} clusters, {np.sum(labels==-1)} noise')
    ax.grid(True, alpha=0.3)
    return ax

fig, ax = plt.subplots(figsize=(10, 6))
plot_dbscan(X_moons, db, f'DBSCAN (eps={db.eps}, min_samples={db.min_samples})', ax)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

## 3. Effect of Epsilon

In [ ]:
eps_values = [0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, eps in zip(axes.ravel(), eps_values):
    db_temp = DBSCAN(eps=eps, min_samples=5)
    db_temp.fit(X_moons)
    plot_dbscan(X_moons, db_temp, f'eps={eps}', ax)

plt.suptitle('DBSCAN: Effect of Epsilon Parameter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. DBSCAN vs K-Means: Where DBSCAN Shines

In [ ]:
from sklearn.datasets import make_circles, make_blobs
from sklearn.cluster import KMeans as SkKMeans

# Generate datasets where DBSCAN excels
datasets = {
    'Moons': make_moons(300, noise=0.08, random_state=42),
    'Circles': make_circles(300, noise=0.05, factor=0.3, random_state=42),
    'Blobs with noise': None,  # will create below
}

# Blobs with noise points
X_b, y_b = make_blobs(250, centers=3, cluster_std=0.5, random_state=42)
noise = np.random.uniform(-10, 10, (50, 2))
X_noisy = np.vstack([X_b, noise])
y_noisy = np.concatenate([y_b, -np.ones(50)])
datasets['Blobs with noise'] = (X_noisy, y_noisy)

fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for col, (name, (X_d, y_d)) in enumerate(datasets.items()):
    n_true = len(np.unique(y_d[y_d >= 0]))
    
    # Ground truth
    ax = axes[0, col]
    scatter = ax.scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap='Set1', alpha=0.6, s=15)
    ax.set_title(f'{name}: Ground Truth')
    ax.grid(True, alpha=0.3)
    
    # K-means
    ax = axes[1, col]
    km = SkKMeans(n_clusters=n_true, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_d)
    ax.scatter(X_d[:, 0], X_d[:, 1], c=km_labels, cmap='Set1', alpha=0.6, s=15)
    ax.set_title(f'K-Means (K={n_true})')
    ax.grid(True, alpha=0.3)
    
    # DBSCAN
    ax = axes[2, col]
    # Choose eps based on data scale
    eps_map = {'Moons': 0.15, 'Circles': 0.15, 'Blobs with noise': 0.8}
    db_temp = DBSCAN(eps=eps_map[name], min_samples=5)
    db_temp.fit(X_d)
    plot_dbscan(X_d, db_temp, f'DBSCAN (eps={eps_map[name]})', ax)

axes[0, 0].set_ylabel('Ground Truth', fontsize=12)
axes[1, 0].set_ylabel('K-Means', fontsize=12)
axes[2, 0].set_ylabel('DBSCAN', fontsize=12)

plt.suptitle('DBSCAN vs K-Means: Non-Convex and Noisy Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. How to Choose Epsilon: K-Distance Plot

Plot the k-th nearest neighbor distance (sorted) for each point. The "elbow" suggests a good epsilon value.

In [ ]:
def k_distance_plot(X, k=5):
    """Plot k-distance for choosing epsilon."""
    sq = np.sum(X ** 2, axis=1)
    dist_sq = sq[:, np.newaxis] + sq[np.newaxis, :] - 2 * X @ X.T
    dists = np.sqrt(np.maximum(dist_sq, 0))
    
    # k-th nearest neighbor distance (excluding self)
    sorted_dists = np.sort(dists, axis=1)
    k_dists = sorted_dists[:, k]  # k-th nearest
    k_dists_sorted = np.sort(k_dists)[::-1]
    
    return k_dists_sorted

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (X_data, name) in zip(axes, [(X_moons, 'Moons'), (X_noisy, 'Blobs+Noise')]):
    k_dists = k_distance_plot(X_data, k=5)
    ax.plot(k_dists, 'b-', linewidth=1.5)
    ax.set_xlabel('Points (sorted)')
    ax.set_ylabel('5-Distance')
    ax.set_title(f'K-Distance Plot for {name}\n(Elbow = good epsilon)')
    ax.grid(True, alpha=0.3)
    
    # Annotate approximate elbow
    if name == 'Moons':
        ax.axhline(y=0.15, color='red', linestyle='--', alpha=0.5, label='eps=0.15')
    else:
        ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='eps=0.8')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Effect of min_samples

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, min_samp in zip(axes, [2, 5, 10, 20]):
    db_temp = DBSCAN(eps=0.2, min_samples=min_samp)
    db_temp.fit(X_moons)
    plot_dbscan(X_moons, db_temp, f'min_samples={min_samp}', ax)

plt.suptitle('Effect of min_samples (eps=0.2)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. DBSCAN Limitation: Varying Density

DBSCAN uses a single global epsilon. If clusters have very different densities, a single epsilon cannot work well for all clusters.

In [ ]:
# Varying density clusters
np.random.seed(42)
X_dense = np.random.randn(200, 2) * 0.3 + [0, 0]
X_sparse = np.random.randn(200, 2) * 1.5 + [6, 0]
X_varying = np.vstack([X_dense, X_sparse])
y_varying = np.array([0]*200 + [1]*200)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Ground truth
axes[0].scatter(X_varying[:200, 0], X_varying[:200, 1], c='blue', alpha=0.5, s=15)
axes[0].scatter(X_varying[200:, 0], X_varying[200:, 1], c='red', alpha=0.5, s=15)
axes[0].set_title('Ground Truth (different densities)')
axes[0].grid(True, alpha=0.3)

for ax, eps in zip(axes[1:], [0.3, 0.8, 1.5]):
    db_temp = DBSCAN(eps=eps, min_samples=5)
    db_temp.fit(X_varying)
    plot_dbscan(X_varying, db_temp, f'eps={eps}', ax)

plt.suptitle('DBSCAN Limitation: Varying Density Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Basic HDBSCAN Intuition

HDBSCAN addresses the varying-density problem by running DBSCAN over a range of epsilon values and building a cluster hierarchy. Here we show the basic intuition: how clusters form and merge as epsilon increases.

In [ ]:
def hdbscan_intuition(X, eps_range, min_samples=5):
    """Show how clusters evolve over a range of epsilon values."""
    results = []
    for eps in eps_range:
        db_temp = DBSCAN(eps=eps, min_samples=min_samples)
        db_temp.fit(X)
        results.append({
            'eps': eps,
            'n_clusters': db_temp.n_clusters_,
            'n_noise': np.sum(db_temp.labels_ == -1),
            'labels': db_temp.labels_.copy()
        })
    return results

eps_range = np.arange(0.05, 2.0, 0.05)
results = hdbscan_intuition(X_varying, eps_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([r['eps'] for r in results], [r['n_clusters'] for r in results], 'b-o', markersize=3, linewidth=2)
axes[0].set_xlabel('Epsilon')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Clusters vs Epsilon')
axes[0].grid(True, alpha=0.3)

axes[1].plot([r['eps'] for r in results], [r['n_noise'] for r in results], 'r-o', markersize=3, linewidth=2)
axes[1].set_xlabel('Epsilon')
axes[1].set_ylabel('Number of Noise Points')
axes[1].set_title('Noise Points vs Epsilon')
axes[1].grid(True, alpha=0.3)

plt.suptitle('HDBSCAN Intuition: How Clusters Evolve with Epsilon', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Show snapshots
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
snapshot_eps = [0.1, 0.3, 0.6, 1.0, 1.5]
for ax, target_eps in zip(axes, snapshot_eps):
    idx = np.argmin(np.abs(eps_range - target_eps))
    r = results[idx]
    db_snap = DBSCAN(eps=r['eps'], min_samples=5)
    db_snap.fit(X_varying)
    plot_dbscan(X_varying, db_snap, f'eps={r["eps"]:.2f}', ax)

plt.suptitle('DBSCAN Snapshots at Different Epsilon Values', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Compare with sklearn

In [ ]:
from sklearn.cluster import DBSCAN as SkDBSCAN
from sklearn.metrics import adjusted_rand_score

# Compare on moons data
our_db = DBSCAN(eps=0.2, min_samples=5)
our_db.fit(X_moons)

sk_db = SkDBSCAN(eps=0.2, min_samples=5)
sk_labels = sk_db.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=our_db.labels_, cmap='Set1', alpha=0.6, s=20)
noise_mask = our_db.labels_ == -1
axes[0].scatter(X_moons[noise_mask, 0], X_moons[noise_mask, 1], c='gray', marker='x', s=30)
axes[0].set_title(f'Our DBSCAN ({our_db.n_clusters_} clusters, {noise_mask.sum()} noise)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=sk_labels, cmap='Set1', alpha=0.6, s=20)
sk_noise = sk_labels == -1
axes[1].scatter(X_moons[sk_noise, 0], X_moons[sk_noise, 1], c='gray', marker='x', s=30)
n_sk_clusters = len(set(sk_labels) - {-1})
axes[1].set_title(f'sklearn DBSCAN ({n_sk_clusters} clusters, {sk_noise.sum()} noise)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Our DBSCAN vs sklearn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Agreement
ari = adjusted_rand_score(our_db.labels_, sk_labels)
print(f"ARI between our labels and sklearn: {ari:.4f}")
print(f"Our clusters: {our_db.n_clusters_}, sklearn clusters: {n_sk_clusters}")
print(f"Our noise: {noise_mask.sum()}, sklearn noise: {sk_noise.sum()}")

---

## Interview Questions

### "DBSCAN vs K-means -- when to use which?"
| | K-Means | DBSCAN |
|---|---|---|
| Cluster shape | Convex (spherical) | Arbitrary |
| Requires K | Yes | No |
| Handles noise | No | Yes (built-in) |
| Cluster sizes | Assumes similar | Any |
| Scalability | O(nkt) -- fast | O(n^2) or O(n log n) with indexing |
| Best for | Well-separated blobs | Complex shapes, outlier detection |

### "How to choose epsilon?"
1. **K-distance plot:** Plot sorted k-th nearest neighbor distances, look for the elbow
2. **Rule of thumb:** `min_samples = 2 * n_features`, then find eps from k-distance plot with k=min_samples
3. **Domain knowledge:** If you know the expected density or minimum cluster size
4. **Use HDBSCAN:** Avoids epsilon selection entirely

### "Time complexity?"
- Naive: $O(n^2)$ for pairwise distances
- With spatial indexing (KD-tree, ball tree): $O(n \log n)$ average case
- Space: $O(n^2)$ for distance matrix (or $O(n)$ with indexing)

### "What about varying density?"
DBSCAN with a single epsilon struggles. Solutions:
- **HDBSCAN:** Hierarchical DBSCAN, varies epsilon and extracts stable clusters
- **OPTICS:** Ordering Points To Identify the Clustering Structure -- creates a reachability plot that can be cut at different levels
- **Adaptive methods:** Local density estimation per region

### Notes
- **OPTICS** produces a reachability plot (like a hierarchical dendrogram) that can be cut at different thresholds to get clusters at different density levels.
- DBSCAN is **not** suitable for high-dimensional data (curse of dimensionality -- distances become meaningless). Use dimensionality reduction first.
- DBSCAN results are sensitive to `eps` but relatively robust to `min_samples`.

In [ ]:
print("Notebook 13 complete: DBSCAN from Scratch")
print("="*50)
print("Implemented:")
print("  - DBSCAN with epsilon-neighborhood")
print("  - Core/border/noise classification")
print("  - Region growing / cluster expansion")
print("  - K-distance plot for epsilon selection")
print("  - HDBSCAN intuition (varying epsilon)")
print("  - DBSCAN vs K-means comparison")
print("  - Validated against sklearn")